In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from shared_utils import *
import monai
from monai.networks.nets import SwinUNETR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'MONAI: {monai.__version__}')


In [ ]:
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
BASE_DIR  = DATA_ROOT / "segformer/experiments/swinunetr_baseline"
BASE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {**SHARED_CONFIG,
    "model_name": "swinunetr",
    "feature_size": 48,
    "n_epochs": 120,
    "patience": 30,
    "swa_start": 40,
    "lr": 3e-5,               # Pretrained transformer için daha güvenli LR
    "weight_decay": 1e-2,     # Regularizasyon artırıldı
    "warmup_epochs": 5,
    "min_sens_floor": 0.80,
    "min_spec_floor": 0.50,
    "sensitivity_first": True,
}
print(f"Ortak veriseti dosyaları (Stratified) kullanılıyor: {DATA_ROOT / "segformer/datas"}")
test_df = pd.read_csv(DATA_ROOT / "segformer/datas" / "external_test_set.csv")
print(f"External Test: {len(test_df)}")


In [ ]:
# ============================================================
# SwinUNETR3DClassifier — Q1 SÜRÜM
# Değişiklik: Tüm 5 katman (1488 dim) YERİNE sadece son 2 katman
# Layer 3 (384 dim) + Layer 4 (768 dim) = 1152 dim
# Neden: 132 hasta ile 1488 dim head overfitting'e mahkum
# Daha az parametre → daha iyi generalizasyon
# ============================================================
class SwinUNETR3DClassifier(nn.Module):
    def __init__(self, num_classes=2, in_channels=1, feature_size=48):
        super().__init__()
        self.backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=14,
            feature_size=feature_size,
            use_checkpoint=True,
            spatial_dims=3,
        )
        # Sadece son 2 SwinViT katmanı:
        # Layer 3: feature_size * 8 = 384
        # Layer 4: feature_size * 16 = 768
        # Toplam: 1152 dim (1488 yerine — %22 daha az parametre)
        dim3 = feature_size * 8   # 384
        dim4 = feature_size * 16  # 768
        total_dim = dim3 + dim4   # 1152

        self.gap = nn.AdaptiveAvgPool3d(1)
        self.gmp = nn.AdaptiveMaxPool3d(1)  # Max pooling de ekle: peak signal yakalanır

        # GAP + GMP concat → 2× genişleme → BN ile stabil
        fused_dim = total_dim * 2  # 2304
        self.bn = nn.BatchNorm1d(fused_dim)

        # Basit, overfit etmeyecek head:
        # BN → Dropout(0.60) → 256 → GELU → Dropout(0.40) → 2
        self.classifier = nn.Sequential(
            nn.Dropout(0.60),
            nn.Linear(fused_dim, 256),
            nn.GELU(),
            nn.Dropout(0.40),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        # hidden: 5 katman [0..4]
        hidden = self.backbone.swinViT(x, self.backbone.normalize)

        # Sadece son 2 katmanı kullan
        feat3 = hidden[3]  # [B, 384, D/16, H/16, W/16]
        feat4 = hidden[4]  # [B, 768, D/32, H/32, W/32]

        # GAP + GMP her katman için
        gap3 = self.gap(feat3).flatten(1)  # [B, 384]
        gmp3 = self.gmp(feat3).flatten(1)  # [B, 384]
        gap4 = self.gap(feat4).flatten(1)  # [B, 768]
        gmp4 = self.gmp(feat4).flatten(1)  # [B, 768]

        # Concat: [GAP3, GMP3, GAP4, GMP4] = 2304 dim
        feat = torch.cat([gap3, gmp3, gap4, gmp4], dim=1)
        feat = self.bn(feat)

        return self.classifier(feat)


In [ ]:
def load_pretrained_swin(model, ckpt_path=None):
    """MONAI SSL pretrained ağırlıklarını yükle."""
    if ckpt_path is None:
        import os
        candidates = [
            os.path.expanduser("~/models/model_swinvit.pt"),
            os.path.expanduser("~/.cache/torch/hub/checkpoints/model_swinvit.pt"),
            str(DATA_ROOT / "segformer/model_swinvit.pt"),
        ]
        ckpt_path = next((p for p in candidates if Path(p).exists()), None)

    if ckpt_path is None or not Path(ckpt_path).exists():
        print("[WARN] Pretrained ağırlık bulunamadı. Random init ile devam.")
        return model

    state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    target = model.backbone.swinViT.state_dict()
    loaded, skipped = 0, 0
    new_sd = {}
    for k, v in state_dict.items():
        k2 = k.replace("swinViT.", "").replace("module.", "")
        if k2 in target and target[k2].shape == v.shape:
            new_sd[k2] = v
            loaded += 1
        else:
            skipped += 1
    model.backbone.swinViT.load_state_dict(new_sd, strict=False)
    print(f"  Pretrained: {loaded} katman yüklendi, {skipped} atlandı.")
    return model


In [ ]:
# ClinicalFocalLoss from shared_utils (pos_weight=2.5, gamma=3.0)
# Bu loss doğrudan tümörü kaçırmayı (FN) cezalandırır
criterion_cls = ClinicalFocalLoss(
    pos_weight=CONFIG.get("pos_weight", 2.5),
    gamma=CONFIG.get("focal_gamma", 3.0),
    smoothing=CONFIG.get("label_smoothing", 0.05),
)
print(f"Loss: ClinicalFocalLoss | pos_weight={CONFIG.get('pos_weight', 2.5)} gamma={CONFIG.get('focal_gamma', 3.0)}")


In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR

def run_one_fold_swin(train_df, val_df, fold_idx, config, output_dir):
    def set_seed(s):
        torch.manual_seed(s); np.random.seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    set_seed(config["random_seed"] + fold_idx)

    fold_dir = Path(output_dir) / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_ds = AppendixH5Dataset(train_df, augment=True,  config=config)
    val_ds   = AppendixH5Dataset(val_df,   augment=False, config=config)

    train_labels = train_df["label"].values.astype(int)
    class_counts = np.bincount(train_labels)
    print(f"  [Fold {fold_idx}] class_counts={class_counts} | Uniform Shuffle kullanılıyor")

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"],
                              shuffle=True, num_workers=config["num_workers"],
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config["batch_size"],
                              shuffle=False, num_workers=config["num_workers"],
                              pin_memory=True)

    model = SwinUNETR3DClassifier(feature_size=config["feature_size"]).to(DEVICE)
    model = load_pretrained_swin(model)

    # Bias başlatma: prior log(pos/neg)
    n_neg, n_pos = class_counts[0], class_counts[1]
    prior_bias = float(np.log(n_pos / (n_neg + 1e-9)))
    with torch.no_grad():
        last_linear = model.classifier[-1]
        nn.init.xavier_uniform_(last_linear.weight)
        last_linear.bias.data[0] = -prior_bias
        last_linear.bias.data[1] =  prior_bias
        print(f"  [Bias init] prior_bias={prior_bias:.3f} (log pos/neg)")

    # pos_weight=1.0: sınıflar dengeli (86/79), focal_gamma loss zaten FN'i cezalandırıyor
    criterion = ClinicalFocalLoss(
        pos_weight=config.get("pos_weight", 1.0),
        gamma=config.get("focal_gamma", 2.0),
        smoothing=config.get("label_smoothing", 0.05)
    )

    # Tüm parametreler açık — backbone baştan öğreniyor
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = get_warmup_cosine_scheduler(
        optimizer, config["warmup_epochs"], config["n_epochs"])

    swa_model = AveragedModel(model)
    swa_start = config.get("swa_start", 50)
    swa_scheduler = SWALR(optimizer, swa_lr=config["lr"] * 0.1)

    best_score   = -1.0
    best_metrics = None
    patience_cnt = 0
    history      = []

    for epoch in range(1, config["n_epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_auc, val_acc, val_f1, pred_df = evaluate_model(
            model, val_loader, criterion, DEVICE)

        y_true = pred_df["label"].values
        y_prob = pred_df["prob_mucinous"].values
        threshold = float(pred_df["_threshold_used"].iloc[0]) \
            if "_threshold_used" in pred_df.columns else 0.5
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        sens = tp / (tp + fn + 1e-9)
        spec = tn / (tn + fp + 1e-9)
        from sklearn.metrics import f1_score as _f1
        val_f1_thr = float(_f1(y_true, y_pred, zero_division=0))
        composite  = clinical_composite(sens, val_auc, spec, f1=val_f1_thr)

        print(f"    [Prob dist] min={y_prob.min():.3f} max={y_prob.max():.3f} mean={y_prob.mean():.3f} | TP={tp} FP={fp} FN={fn} TN={tn}")
        print(f"[swinunetr | fold {fold_idx} | epoch {epoch:03d}] "
              f"train={train_loss:.4f} val_loss={val_loss:.4f} "
              f"auc={val_auc:.4f} acc={val_acc:.4f} thr={threshold:.3f}")
        print(f"  [Metrics] AUC:{val_auc:.3f} F1:{val_f1_thr:.3f} "
              f"SENS:{sens:.3f} SPEC:{spec:.3f} | COMPOSITE:{composite:.4f}")

        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "auc": val_auc,
                        "sens": sens, "spec": spec, "f1": val_f1_thr,
                        "composite": composite})

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        # ── ÇİFT KISIT model seçimi: SENS≥0.80 AND SPEC≥0.50 ────────
        min_sens_floor = config.get("min_sens_floor", 0.80)
        min_spec_floor = config.get("min_spec_floor", 0.50)
        sens_ok = (sens >= min_sens_floor)
        spec_ok = (spec >= min_spec_floor)
        if sens_ok and spec_ok and composite > best_score:
            best_score   = composite
            best_metrics = {"auc": val_auc, "sens": sens, "spec": spec,
                            "f1": val_f1_thr, "composite": composite, "epoch": epoch}
            torch.save(model.state_dict(), fold_dir / "best_model.pt")
            patience_cnt = 0
            print(f"    ★ Best model saved (SENS={sens:.3f}≥{min_sens_floor} "
                  f"SPEC={spec:.3f}≥{min_spec_floor} | F1={val_f1_thr:.3f} | composite={composite:.4f})")
        else:
            patience_cnt += 1
            if patience_cnt >= config["patience"]:
                print(f"  Early stopping @ epoch {epoch} | best_score={best_score:.4f}")
                break

    # SWA finalize
    if epoch >= swa_start:
        print("  SWA model finalize ediliyor...")
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
        _, val_auc_swa, _, _, pred_df_swa = evaluate_model(swa_model, val_loader, criterion, DEVICE)
        y_t = pred_df_swa["label"].values; y_p = pred_df_swa["prob_mucinous"].values
        thr_s = float(pred_df_swa["_threshold_used"].iloc[0])
        yp_s  = (y_p >= thr_s).astype(int)
        tn_, fp_, fn_, tp_ = confusion_matrix(y_t, yp_s, labels=[0,1]).ravel()
        swa_comp = clinical_composite(
            tp_/(tp_+fn_+1e-9), val_auc_swa, tn_/(tn_+fp_+1e-9),
            f1=float(_f1(y_t, yp_s, zero_division=0)))
        print(f"  SWA COMPOSITE: {swa_comp:.4f} | Best regular: {best_score:.4f}")
        if swa_comp > best_score:
            torch.save(swa_model.state_dict(), fold_dir / "best_model.pt")
            print("  SWA modeli seçildi!")

    # Hiç model kaydedilmediyse fallback
    if best_score < 0:
        print("  [WARN] Çift kısıt hiç sağlanamadı — son epoch fallback olarak kaydediliyor")
        torch.save(model.state_dict(), fold_dir / "best_model.pt")

    # Final evaluation
    model.load_state_dict(torch.load(fold_dir / "best_model.pt", map_location=DEVICE, weights_only=False))
    _, val_auc_f, _, _, pred_df_f = evaluate_model(model, val_loader, criterion, DEVICE)
    youden_thr, _ = find_youden_threshold(pred_df_f["label"].values, pred_df_f["prob_mucinous"].values)
    ci = compute_bootstrap_ci(pred_df_f["label"].values, pred_df_f["prob_mucinous"].values, youden_thr)
    metrics_f, cm_f, _ = compute_binary_metrics(
        pred_df_f["label"].values, pred_df_f["prob_mucinous"].values, youden_thr)

    print_full_metrics_table(metrics_f, ci, f"SwinUNETR Fold {fold_idx}", f"Youden {youden_thr:.3f}")
    plot_confusion_matrix(cm_f, f"Fold {fold_idx} Confusion Matrix",
                          save_path=fold_dir / f"cm_fold{fold_idx}.png")
    pred_df_f.to_csv(fold_dir / "val_predictions.csv", index=False)

    import json as _json
    with open(fold_dir / "history.json", "w") as fh:
        _json.dump(history, fh)

    return metrics_f, ci, pred_df_f, history


In [ ]:
import json

all_preds, all_metrics = [], []

for fold_idx in range(1, 6):
    train_df = pd.read_csv(DATA_ROOT / "segformer/datas" / f"fold_{fold_idx}_train.csv")
    val_df   = pd.read_csv(DATA_ROOT / "segformer/datas" / f"fold_{fold_idx}_val.csv")

    print(f"{'='*70}FOLD {fold_idx}/5{'='*70}")
    m_f, ci_f, pred_f, hist_f = run_one_fold_swin(train_df, val_df, fold_idx, CONFIG, BASE_DIR)
    all_preds.append(pred_f)
    all_metrics.append(m_f)

print("5-Fold CV tamamlandı.")


In [ ]:
# ── External Test: Her Fold Modeli Ayrı Ayrı + Ensemble ──────────
test_ds = AppendixH5Dataset(test_df, augment=False, config=CONFIG)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"],
                         shuffle=False, num_workers=CONFIG["num_workers"])

fold_probs_ext = []
fold_rows_ext  = []
criterion_eval = ClinicalFocalLoss()

print(f"EXTERNAL TEST (ALL FOLDS + ENSEMBLE):")
for fold_idx in range(1, 6):
    fold_dir = BASE_DIR / f"fold_{fold_idx}"
    model_e = SwinUNETR3DClassifier(feature_size=CONFIG["feature_size"]).to(DEVICE)
    model_e.load_state_dict(torch.load(fold_dir / "best_model.pt", map_location=DEVICE, weights_only=False))
    _, _, _, _, pred_ext = evaluate_model(model_e, test_loader, criterion_eval, DEVICE)
    del model_e; torch.cuda.empty_cache()
    
    y_true_e = pred_ext["label"].values
    y_prob_e = pred_ext["prob_mucinous"].values
    fold_probs_ext.append(y_prob_e)
    m_e, _, _ = compute_binary_metrics(y_true_e, y_prob_e, SHARED_CONFIG["default_threshold"])
    fold_rows_ext.append({"fold": f"Fold {fold_idx}",
                          "auc_roc": round(m_e["auc_roc"],3),
                          "sensitivity": round(m_e["sensitivity"],3),
                          "specificity": round(m_e["specificity"],3),
                          "accuracy": round(m_e["accuracy"],3),
                          "f1": round(m_e["f1"],3),
                          "tp": m_e["tp"], "fp": m_e["fp"],
                          "fn": m_e["fn"], "tn": m_e["tn"]})

# Ensemble
ens_prob = np.mean(fold_probs_ext, axis=0)
y_true_ext = test_df["label"].values[:len(ens_prob)]

youden_t, _ = find_youden_threshold(y_true_ext, ens_prob)
sens_t = find_sensitivity_threshold(y_true_ext, ens_prob, min_sensitivity=0.90)
m_youden, cm_y, _ = compute_binary_metrics(y_true_ext, ens_prob, youden_t)
m_05,     cm_05,_ = compute_binary_metrics(y_true_ext, ens_prob, 0.5)
m_sens,   cm_s, _ = compute_binary_metrics(y_true_ext, ens_prob, sens_t)

fold_rows_ext += [
    {"fold": "Ensemble (@Youden)",  **{k: round(v,3) if isinstance(v,float) else v
      for k,v in m_youden.items() if k in ["auc_roc","sensitivity","specificity","accuracy","f1","tp","fp","fn","tn"]}},
    {"fold": "Ensemble (@0.5)",     **{k: round(v,3) if isinstance(v,float) else v
      for k,v in m_05.items()     if k in ["auc_roc","sensitivity","specificity","accuracy","f1","tp","fp","fn","tn"]}},
    {"fold": "Ensemble (90+ Sens)", **{k: round(v,3) if isinstance(v,float) else v
      for k,v in m_sens.items()   if k in ["auc_roc","sensitivity","specificity","accuracy","f1","tp","fp","fn","tn"]}},
]

print(pd.DataFrame(fold_rows_ext).to_string(index=False))
ci_ens = compute_bootstrap_ci(y_true_ext, ens_prob, youden_t)
print("=== ENSEMBLE @YOUDEN ===")
print_full_metrics_table(m_youden, ci_ens, "SwinUNETR Ensemble", f"Youden {youden_t:.3f}")
print("=== ENSEMBLE @90+ SENS ===")
ci_ens2 = compute_bootstrap_ci(y_true_ext, ens_prob, sens_t)
print_full_metrics_table(m_sens, ci_ens2, "SwinUNETR Ensemble", f"90+Sens {sens_t:.3f}")

# Grafikler
plot_dir = BASE_DIR / "external_test_folds"
plot_dir.mkdir(exist_ok=True)
plot_roc_pr(y_true_ext, ens_prob, "SwinUNETR_Ensemble", plot_dir, opt_threshold=youden_t)
plot_confusion_matrix(cm_y, "Ensemble @Youden", save_path=plot_dir / "cm_ensemble_youden.png")
plot_confusion_matrix(cm_s, "Ensemble @90+Sens", save_path=plot_dir / "cm_ensemble_90sens.png")

# Sonuçları kaydet
results = {"fold_metrics": all_metrics,
           "ensemble_youden": m_youden, "ensemble_90sens": m_sens}
with open(BASE_DIR / "all_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"Sonuçlar kaydedildi: {BASE_DIR / 'all_results.json'}")
